# Fine-tuning mT5-small for English ↔ 11 languages
Training and evaluation for the Real-Time Voice Translation app. Base model google/mt5-small, data OPUS-100, run on a Google Colab T4. Final model: Gunal13/mt5-small-11lang. Training loss is shown about 4× too high (summed over 4 accumulation steps); compare validation loss.

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
!pip install -q datasets sentencepiece sacrebleu

In [5]:
!nvidia-smi

Tue Nov 24 01:26:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Why mT5 and not T5
T5-small's tokenizer turns every Hindi word into <unk>; mT5 covers 101 languages.

In [6]:
from transformers import AutoTokenizer

t5 = AutoTokenizer.from_pretrained("t5-small")
mt5 = AutoTokenizer.from_pretrained("google/mt5-small")

sentence = "आप कैसे हो"   # Hindi: "How are you"
for name, tok in [("t5-small", t5), ("mt5-small", mt5)]:
    ids = tok.encode(sentence, add_special_tokens=False)
    print(name)
    print("  tokens: ", tok.convert_ids_to_tokens(ids))
    print("  unknown:", ids.count(tok.unk_token_id), "of", len(ids))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


t5-small
  tokens:  ['▁', '<unk>', '▁', '<unk>', '▁', '<unk>']
  unknown: 3 of 6
mt5-small
  tokens:  ['▁आप', '▁कै', 'से', '▁हो']
  unknown: 0 of 4


## English–Hindi data (OPUS-100)

In [7]:
from datasets import load_dataset

hi = load_dataset("Helsinki-NLP/opus-100", "en-hi")
print(hi)

for ex in hi["train"].select(range(5)):
    print(ex["translation"]["en"], "  →  ", ex["translation"]["hi"])

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 534319
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
Other, Private Use   →   अन्य, निज़ी उपयोग
[SCREAMING]   →   ऊबड़ .
Spouse   →   जीवनसाथी
I will never salute you!   →   - तुम एक कमांडर कभी नहीं होगा!
and the stars and the trees bow themselves;   →   और तारे और वृक्ष सजदा करते है;


## Cleaning
Six filters; about 1 in 4 raw pairs was a duplicate.

In [8]:
import re
import pandas as pd

# Put the training data into a table
df = pd.DataFrame({
    "en": [t["en"] for t in hi["train"]["translation"]],
    "hi": [t["hi"] for t in hi["train"]["translation"]],
})

def devanagari_ratio(text):
    """Fraction of letters in the text that are Hindi (Devanagari) letters."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum("\u0900" <= c <= "\u097F" for c in letters) / len(letters)

report = [("Start", len(df))]

# 1. Trim spaces and subtitle dashes like "- Hello"
for col in ["en", "hi"]:
    df[col] = df[col].str.strip().str.lstrip("- ").str.strip()
df = df[(df.en != "") & (df.hi != "")]
report.append(("Remove empty", len(df)))

# 2. Remove subtitle tags and symbols like [SCREAMING], ♪, <i>
tags = r"[\[\]♪#<>{}]"
df = df[~df.en.str.contains(tags) & ~df.hi.str.contains(tags)]
report.append(("Remove tags/symbols", len(df)))

# 3. English side must be English; Hindi side must be mostly Hindi
df = df[df.en.str.contains(r"[A-Za-z]") & ~df.en.str.contains(r"[\u0900-\u097F]")]
df = df[df.hi.apply(devanagari_ratio) >= 0.7]
report.append(("Check languages", len(df)))

# 4. Remove very long sentences (over 50 words)
df = df[(df.en.str.split().str.len() <= 50) & (df.hi.str.split().str.len() <= 50)]
report.append(("Remove long sentences", len(df)))

# 5. Remove pairs whose lengths don't match (likely misaligned)
ratio = df.hi.str.len() / df.en.str.len()
df = df[(ratio >= 0.5) & (ratio <= 2.0)]
report.append(("Remove length mismatch", len(df)))

# 6. Remove duplicate pairs
df = df.drop_duplicates(subset=["en", "hi"]).reset_index(drop=True)
report.append(("Remove duplicates", len(df)))

# Show what each step removed
prev = report[0][1]
for step, n in report:
    print(f"{step:25s} {n:>8,}   (removed {prev - n:>7,})")
    prev = n
print(f"\nKept {len(df) / report[0][1]:.1%} of the original data")

print("\n10 random cleaned examples:")
for _, row in df.sample(10, random_state=1).iterrows():
    print(row.en, "  →  ", row.hi)

Start                      534,319   (removed       0)
Remove empty               534,286   (removed      33)
Remove tags/symbols        518,023   (removed  16,263)
Check languages            477,218   (removed  40,805)
Remove long sentences      453,350   (removed  23,868)
Remove length mismatch     434,909   (removed  18,441)
Remove duplicates          294,690   (removed 140,219)

Kept 55.2% of the original data

10 random cleaned examples:
Well, they haven't responded as of yet.   →   खैर, वे नहीं है अभी तक के रूप में यह जवाब दिया.
but whoever goes beyond this is a transgressor;   →   किन्तु जिस किसी ने इसके अतिरिक्त कुछ और चाहा तो ऐसे ही लोग सीमा का उल्लंघन करनेवाले है।-
Nay, but (man) hath not done what He commanded him.   →   कदापि नहीं, उसने उसको पूरा नहीं किया जिसका आदेश अल्लाह ने उसे दिया है
If any, after this, invent a lie and attribute it to Allah, they are indeed unjust wrong-doers.   →   अब इसके पश्चात भी जो व्यक्ति झूठी बातें अल्लाह से जोड़े, तो ऐसे ही लोग अत्याचारी है
On

## Training examples
100k pairs, each used in both directions.

In [9]:
import numpy as np
from datasets import Dataset

tok = mt5  # the mT5 tokenizer from Cell 3

def make_examples(frame):
    """Each pair becomes two examples: English→Hindi and Hindi→English."""
    src = (["translate English to Hindi: " + s for s in frame.en] +
           ["translate Hindi to English: " + s for s in frame.hi])
    tgt = list(frame.hi) + list(frame.en)
    return Dataset.from_dict({"src": src, "tgt": tgt}).shuffle(seed=42)

# Training data: 100k cleaned pairs → 200k examples
train_df = df.sample(n=100_000, random_state=42).reset_index(drop=True)
train_ds = make_examples(train_df)

# Validation data: the official 2,000-pair validation set (both directions)
val_df = pd.DataFrame({
    "en": [t["en"] for t in hi["validation"]["translation"]],
    "hi": [t["hi"] for t in hi["validation"]["translation"]],
})
val_ds = make_examples(val_df)

print(train_ds)
print(val_ds)
print("\nExample:", train_ds[0])

# How long are the sentences in tokens? This decides the max length.
lens = [len(tok(s).input_ids) for s in train_ds.select(range(5000))["src"]]
p50, p90, p95, p99 = np.percentile(lens, [50, 90, 95, 99])
print(f"\nToken lengths: median {p50:.0f}, 90% {p90:.0f}, 95% {p95:.0f}, 99% {p99:.0f}")

Dataset({
    features: ['src', 'tgt'],
    num_rows: 200000
})
Dataset({
    features: ['src', 'tgt'],
    num_rows: 4000
})

Example: {'src': 'translate Hindi to English: डेविड पूछो कि क्या पासपोर्ट जारी कर दिया गया है या नहीं।', 'tgt': "Ask David whether the passport's been issued or not."}

Token lengths: median 22, 90% 61, 95% 74, 99% 100


## Tokenize and check mT5 before training

In [10]:
import torch
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MAX_LEN = 128

def tokenize(batch):
    """Turn text into token IDs; the target becomes 'labels'."""
    enc = tok(batch["src"], max_length=MAX_LEN, truncation=True)
    enc["labels"] = tok(text_target=batch["tgt"], max_length=MAX_LEN,
                        truncation=True)["input_ids"]
    return enc

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["src", "tgt"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["src", "tgt"])

# Load pretrained mT5-small
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to("cuda")
print(f"Model size: {model.num_parameters() / 1e6:.0f}M parameters")

# Pads each batch to its own longest sentence; pads labels with -100 so padding is ignored in the loss
collator = DataCollatorForSeq2Seq(tok, model=model)

# BEFORE training: what does mT5 output?
tests = ["translate English to Hindi: How are you?",
         "translate Hindi to English: आप कैसे हो"]
enc = tok(tests, return_tensors="pt", padding=True).to("cuda")
out = model.generate(**enc, max_new_tokens=20)
print("\nBefore training:")
for s, o in zip(tests, tok.batch_decode(out, skip_special_tokens=False)):
    print(f"  {s}\n    → {o}")

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model size: 300M parameters

Before training:
  translate English to Hindi: How are you?
    → <pad> <extra_id_0></s>
  translate Hindi to English: आप कैसे हो
    → <pad> <extra_id_0></s>


## Speed and memory test
Batch 32 with AdamW ran out of memory, so: batch 8 × gradient accumulation 4 with Adafactor, fp32.

In [12]:
import gc, time
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer



# Fresh mT5 (the failed run may have partly changed the old one)
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to("cuda")
collator = DataCollatorForSeq2Seq(tok, model=model)

BATCH = 8
ACCUM = 4   # 8 x 4 = effective batch of 32

args = Seq2SeqTrainingArguments(
    output_dir="speed_test",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    optim="adafactor",
    learning_rate=5e-4,
    max_steps=100,
    logging_steps=20,
    fp16=False,          # mT5 gives NaN loss in 16-bit
    report_to="none",
    save_strategy="no",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    data_collator=collator,
    processing_class=tok,
)

start = time.time()
trainer.train()
sec_per_step = (time.time() - start) / 100

steps_per_epoch = len(train_tok) // (BATCH * ACCUM)
print(f"\n{sec_per_step:.2f} seconds per step")
print(f"One epoch = {steps_per_epoch:,} steps ≈ {sec_per_step * steps_per_epoch / 60:.0f} minutes")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB of 15 GB")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss
20,58.566919
40,25.760071
60,22.170035
80,20.444051
100,19.803166



1.12 seconds per step
One epoch = 6,250 steps ≈ 116 minutes
Peak GPU memory: 6.2 GB of 15 GB


## Log in to Hugging Face
The token is typed into a hidden box and not saved.

In [13]:

from getpass import getpass
from huggingface_hub import login, whoami

login(token=getpass("Paste your Hugging Face token: "))
print("Logged in as:", whoami()["name"])

Logged in as: Gunal13


## Stage 1: train English ↔ Hindi

In [14]:
import gc
from huggingface_hub import whoami

# Clear the speed-test model and load a fresh mT5
del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to("cuda")
collator = DataCollatorForSeq2Seq(tok, model=model)

repo_id = f"{whoami()['name']}/mt5-small-en-hi"

args = Seq2SeqTrainingArguments(
    output_dir="mt5-small-en-hi",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,     # effective batch 32
    per_device_eval_batch_size=16,
    optim="adafactor",
    learning_rate=5e-4,
    warmup_steps=500,                  # start with small steps, then full speed
    fp16=False,                        # mT5 gives NaN in 16-bit

    eval_strategy="steps",             # check validation loss...
    eval_steps=1000,                   # ...every 1,000 steps
    save_strategy="steps",
    save_steps=1000,                   # save a checkpoint every 1,000 steps
    save_total_limit=2,                # keep only the 2 newest locally
    load_best_model_at_end=True,       # at the end, keep the checkpoint with lowest validation loss
    metric_for_best_model="eval_loss",
    logging_steps=100,

    push_to_hub=True,                  # upload checkpoints to Hugging Face
    hub_model_id=repo_id,
    hub_strategy="checkpoint",         # always keep the latest checkpoint online, for resuming
    hub_private_repo=True,             # private for now; make public once results look good
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    processing_class=tok,
)

trainer.train()
trainer.push_to_hub(commit_message="Finished training: 1 epoch on 100k cleaned OPUS-100 en-hi pairs")
print(f"\nDone! Model saved to https://huggingface.co/{repo_id}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss,Validation Loss
1000,13.516794,3.027099
2000,11.879412,2.780870
3000,11.010145,2.655240
4000,10.599469,2.563401
5000,10.267208,2.510063
6000,10.106691,2.485071
6250,10.140526,2.482764


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Done! Model saved to https://huggingface.co/Gunal13/mt5-small-en-hi


## Quick test after stage 1

In [15]:
model = trainer.model
model.eval()

tests = [
    "translate English to Hindi: How are you?",
    "translate Hindi to English: आप कैसे हो",
    "translate English to Hindi: Where is the train station?",
    "translate English to Hindi: I am learning machine learning at university.",
    "translate English to Hindi: Can you help me with my homework?",
    "translate Hindi to English: मुझे भूख लगी है",      # I am hungry
    "translate Hindi to English: कल बारिश होगी",         # It will rain tomorrow
    "translate Hindi to English: मेरा नाम गुनाल है",     # My name is Gunal
]

enc = tok(tests, return_tensors="pt", padding=True).to("cuda")
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=60, num_beams=4)

for s, o in zip(tests, tok.batch_decode(out, skip_special_tokens=True)):
    print(f"{s}\n  → {o}\n")

translate English to Hindi: How are you?
  → तुम कैसे हो?

translate Hindi to English: आप कैसे हो
  → How are you?

translate English to Hindi: Where is the train station?
  → रेल स्टेशन क्या है?

translate English to Hindi: I am learning machine learning at university.
  → मैं विश्वविद्यालय में प्रयोग कर रहा हूँ.

translate English to Hindi: Can you help me with my homework?
  → क्या तुम मुझे मेरे शिक्षण के साथ help सकते हो?

translate Hindi to English: मुझे भूख लगी है
  → I'm sick.

translate Hindi to English: कल बारिश होगी
  → Tomorrow sunrise will be sunrise.

translate Hindi to English: मेरा नाम गुनाल है
  → I'm my name.



## Evaluation: OPUS-100 test set (2,000 sentences)

In [16]:
import sacrebleu

test_df = pd.DataFrame({
    "en": [t["en"] for t in hi["test"]["translation"]],
    "hi": [t["hi"] for t in hi["test"]["translation"]],
})

def translate_all(sources, batch=32):
    preds = []
    for i in range(0, len(sources), batch):
        enc = tok(sources[i:i + batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=128, num_beams=4)
        preds += tok.batch_decode(out, skip_special_tokens=True)
    return preds

en2hi = translate_all(["translate English to Hindi: " + s for s in test_df.en])
hi2en = translate_all(["translate Hindi to English: " + s for s in test_df.hi])

print("=== Test set results (2,000 sentences each) ===")
for name, preds, refs in [("English → Hindi", en2hi, list(test_df.hi)),
                          ("Hindi → English", hi2en, list(test_df.en))]:
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    print(f"{name}:  BLEU {bleu:.1f}   chrF {chrf:.1f}")

print("\n=== 5 random English → Hindi examples ===")
for i in test_df.sample(5, random_state=7).index:
    print(f"EN:   {test_df.en[i]}\nREF:  {test_df.hi[i]}\nPRED: {en2hi[i]}\n")

=== Test set results (2,000 sentences each) ===
English → Hindi:  BLEU 7.6   chrF 24.1
Hindi → English:  BLEU 10.1   chrF 29.1

=== 5 random English → Hindi examples ===
EN:   - Please, somebody!
REF:  कृपया, किसी को!
PRED: कृपया, कोई भी!

EN:   Yes.
REF:  हाँ।
PRED: हाँ.

EN:   Anyway, art moved on, as it must, it has to.
REF:  वैसे भी, कला पर ले जाया गया, केरूपमेंयहचाहिए ,यहकरनेके लिएहै.
PRED: अच्छा, कला बंद कर दिया जाता है, जैसे यह चाहता है, यह ठीक है.

EN:   And perhaps for the first time there is an online website called No Criminals that is geared towards educating voters about political candidates that have a criminal record and stop them from contesting elections.
REF:  एक अन्य जालस्थल नो क्रिमिनल्स में मतदाताओं को आपराधिक पृष्ठभूमि वाले प्रत्याशियों के बारे में जानकारी दी गई है।
PRED: और अगले समय के लिए एक ऑनलाइन पृष्ठभूमि के बारे में कोई प्रमाणपत्र नहीं था कि राजनीतिक पार्टी के बारे में राजनीतिक प्रतिनिधियों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मत

## Experiment: n-gram blocking
no_repeat_ngram_size=3: +1.5 chrF English→Hindi without retraining.

In [17]:
def translate_all(sources, batch=32, **gen_settings):
    preds = []
    for i in range(0, len(sources), batch):
        enc = tok(sources[i:i + batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=128, num_beams=4, **gen_settings)
        preds += tok.batch_decode(out, skip_special_tokens=True)
    return preds

en2hi_v2 = translate_all(["translate English to Hindi: " + s for s in test_df.en],
                         no_repeat_ngram_size=3)
hi2en_v2 = translate_all(["translate Hindi to English: " + s for s in test_df.hi],
                         no_repeat_ngram_size=3)

print("=== Baseline vs. no-repeat decoding ===")
for name, old, new, refs in [("English → Hindi", en2hi, en2hi_v2, list(test_df.hi)),
                             ("Hindi → English", hi2en, hi2en_v2, list(test_df.en))]:
    b0 = sacrebleu.corpus_bleu(old, [refs]).score
    c0 = sacrebleu.corpus_chrf(old, [refs]).score
    b1 = sacrebleu.corpus_bleu(new, [refs]).score
    c1 = sacrebleu.corpus_chrf(new, [refs]).score
    print(f"{name}:  BLEU {b0:.1f} → {b1:.1f}   chrF {c0:.1f} → {c1:.1f}")

# The long sentence that looped before
i = test_df.index[test_df.en.str.startswith("And perhaps for the first time")][0]
print(f"\nLong sentence, before:\n{en2hi[i]}\n\nAfter:\n{en2hi_v2[i]}")

=== Baseline vs. no-repeat decoding ===
English → Hindi:  BLEU 7.6 → 7.8   chrF 24.1 → 25.6
Hindi → English:  BLEU 10.1 → 9.9   chrF 29.1 → 29.4

Long sentence, before:
और अगले समय के लिए एक ऑनलाइन पृष्ठभूमि के बारे में कोई प्रमाणपत्र नहीं था कि राजनीतिक पार्टी के बारे में राजनीतिक प्रतिनिधियों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब करनेवालों के बारे में मतलब नहीं है और उनके लिए

After:
और अगले समय के लिए एक ऑनलाइन पृष्ठभूमि के बारे में कोई वकील नहीं है, जो राजनीतिक पार्टी के विषय में विरोध करनेवालों के साथ राजनीति करने वाले राजनीतिकों को अपराधियों से रोकता है और उनके लिए चुनावों से हटकर लौट रहा है.


## Stage 2 data: 11 languages

In [19]:
LANGS = {  # code: (OPUS-100 name, language name, script range)
    "hi": ("en-hi", "Hindi",     ("\u0900", "\u097F")),
    "mr": ("en-mr", "Marathi",   ("\u0900", "\u097F")),
    "ta": ("en-ta", "Tamil",     ("\u0B80", "\u0BFF")),
    "kn": ("en-kn", "Kannada",   ("\u0C80", "\u0CFF")),
    "ml": ("en-ml", "Malayalam", ("\u0D00", "\u0D7F")),
    "zh": ("en-zh", "Chinese",   ("\u4E00", "\u9FFF")),
    "af": ("af-en", "Afrikaans", None),   # Latin alphabet, no script check
    "es": ("en-es", "Spanish",   None),
    "te": ("en-te", "Telugu",    ("\u0C00", "\u0C7F")),
    "ar": ("ar-en", "Arabic",    ("\u0600", "\u06FF")),
    "ru": ("en-ru", "Russian",   ("\u0400", "\u04FF")),
}
PER_LANG = 20_000      # training pairs per language
VAL_PER_LANG = 300     # validation pairs per language

def script_ratio(text, rng):
    """Fraction of letters that belong to the given script."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(rng[0] <= c <= rng[1] for c in letters) / len(letters)

def clean_pairs(df, code):
    rng = LANGS[code][2]
    for col in ["en", "tgt"]:
        df[col] = df[col].str.strip().str.lstrip("- ").str.strip()
    df = df[(df.en != "") & (df.tgt != "")]
    df = df[~df.en.str.contains(r"[\[\]♪#<>{}]") & ~df.tgt.str.contains(r"[\[\]♪#<>{}]")]
    df = df[df.en.str.contains(r"[A-Za-z]")]
    df = df[df.en.str.lower() != df.tgt.str.lower()]          # untranslated copies
    if rng:
        df = df[df.en.apply(lambda t: script_ratio(t, rng)) == 0]
        df = df[df.tgt.apply(lambda t: script_ratio(t, rng)) >= 0.7]
    df = df[(df.en.str.split().str.len() <= 50) & (df.tgt.str.len() <= 400)]
    ratio = df.tgt.str.len() / df.en.str.len()
    lo, hi_ = (0.1, 1.5) if code == "zh" else (0.4, 2.5)
    df = df[(ratio >= lo) & (ratio <= hi_)]
    return df.drop_duplicates(subset=["en", "tgt"]).reset_index(drop=True)

def to_frame(split, code, name):
    return pd.DataFrame({
        "en":   [t["en"] for t in split["translation"]],
        "tgt":  [t[code] for t in split["translation"]],
        "lang": name,
    })

train_parts, val_parts, stats = [], [], []
for code, (config, name, rng) in LANGS.items():
    print(f"Loading {name}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config)
    raw = ds["train"].shuffle(seed=42).select(range(min(150_000, len(ds["train"]))))
    clean = clean_pairs(to_frame(raw, code, name), code)

    n_take = min(PER_LANG, 2 * len(clean))   # repeat small languages at most 2x
    if n_take <= len(clean):
        used = clean.sample(n_take, random_state=42)
    else:
        used = pd.concat([clean, clean.sample(n_take - len(clean), random_state=42)])

    train_parts.append(used)
    val_parts.append(to_frame(ds["validation"].select(range(VAL_PER_LANG)), code, name))
    stats.append((name, len(ds["train"]), len(raw), len(clean), len(used)))

train_all = pd.concat(train_parts, ignore_index=True)
val_all = pd.concat(val_parts, ignore_index=True)

def to_examples(frame):
    """Both directions for every language."""
    src = (("translate English to " + frame.lang + ": " + frame.en).tolist() +
           ("translate " + frame.lang + " to English: " + frame.tgt).tolist())
    tgt = frame.tgt.tolist() + frame.en.tolist()
    return Dataset.from_dict({"src": src, "tgt": tgt}).shuffle(seed=42)

ml_train_ds = to_examples(train_all)
ml_val_ds = to_examples(val_all)

print(f"\n{'Language':10s} {'Total':>10s} {'Sampled':>9s} {'Clean':>8s} {'Used':>7s}")
for name, total, sampled, clean_n, used_n in stats:
    print(f"{name:10s} {total:>10,} {sampled:>9,} {clean_n:>8,} {used_n:>7,}")

steps = len(ml_train_ds) // 32
print(f"\nTraining examples: {len(ml_train_ds):,}   Validation: {len(ml_val_ds):,}")
print(f"Estimated training: {steps:,} steps ≈ {steps * 1.12 / 3600:.1f} hours")
print("\nSample:", ml_train_ds[0])

Loading Hindi...
Loading Marathi...
Loading Tamil...
Loading Kannada...
Loading Malayalam...
Loading Chinese...
Loading Afrikaans...
Loading Spanish...
Loading Telugu...


en-te/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.6kB            

en-te/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-te/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.85MB            

en-te/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-te/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  100kB            

... (21 outputs left out)

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]


Language        Total   Sampled    Clean    Used
Hindi         534,319   150,000  103,820  20,000
Marathi        27,007    27,007   20,875  20,000
Tamil         227,014   150,000   97,552  20,000
Kannada        14,537    14,537    9,096  18,192
Malayalam     822,746   150,000  126,941  20,000
Chinese     1,000,000   150,000  128,233  20,000
Afrikaans     275,512   150,000   82,475  20,000
Spanish     1,000,000   150,000  133,300  20,000
Telugu         64,352    64,352   29,050  20,000
Arabic      1,000,000   150,000  135,988  20,000
Russian     1,000,000   150,000  125,040  20,000

Training examples: 436,384   Validation: 6,600
Estimated training: 13,637 steps ≈ 4.2 hours

Sample: {'src': 'translate Tamil to English: நிச்சயமாக, ஜக்கூம் (கள்ளி) மரம் (அதுவே).', 'tgt': 'Surely the tree of the Zaqqum,'}


## Stage 2: train 11 languages
Colab disconnected at step 13,294 of 13,637; checkpoint 13,000 had the lowest validation loss and is the final model.

In [ ]:
import gc

# Free the GPU from the Hindi run
del trainer, model
gc.collect()
torch.cuda.empty_cache()

# Tokenize the 11-language data
ml_train_tok = ml_train_ds.map(tokenize, batched=True, remove_columns=["src", "tgt"])
ml_val_tok = ml_val_ds.map(tokenize, batched=True, remove_columns=["src", "tgt"])

# Start from your English-Hindi model (it already knows how to translate)
user = whoami()["name"]
model = AutoModelForSeq2SeqLM.from_pretrained(f"{user}/mt5-small-en-hi").to("cuda")
collator = DataCollatorForSeq2Seq(tok, model=model)
repo_id = f"{user}/mt5-small-11lang"

args = Seq2SeqTrainingArguments(
    output_dir="mt5-small-11lang",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    optim="adafactor",
    learning_rate=5e-4,
    warmup_steps=500,
    fp16=False,

    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=200,

    push_to_hub=True,
    hub_model_id=repo_id,
    hub_strategy="checkpoint",
    hub_private_repo=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=ml_train_tok,
    eval_dataset=ml_val_tok,
    data_collator=collator,
    processing_class=tok,
)

trainer.train()
trainer.push_to_hub(commit_message="Finished training: 11 languages x ~20k cleaned OPUS-100 pairs, both directions")
print(f"\nDone! Model saved to https://huggingface.co/{repo_id}")

Map:   0%|          | 0/436384 [00:00<?, ? examples/s]

Map:   0%|          | 0/6600 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

... (12 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## New session: recover checkpoint 13,000

In [1]:
import json, torch
from getpass import getpass
from huggingface_hub import login, whoami, hf_hub_download
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

login(token=getpass("Paste your Hugging Face token: "))
user = whoami()["name"]
repo_id = f"{user}/mt5-small-11lang"

# Confirm which checkpoint was saved
state = json.load(open(hf_hub_download(repo_id, "last-checkpoint/trainer_state.json")))
print("Saved checkpoint step:", state["global_step"])
print("Best validation loss:", round(state["best_metric"], 4))

tok = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained(repo_id, subfolder="last-checkpoint").to("cuda")
model.eval()

tests = [
    "translate English to Tamil: How are you?",
    "translate English to Spanish: Where is the train station?",
    "translate English to Russian: I am hungry.",
    "translate English to Arabic: Thank you very much.",
    "translate English to Telugu: Good morning.",
    "translate Spanish to English: ¿Dónde está la estación de tren?",
    "translate Russian to English: Я люблю программирование.",
    "translate Hindi to English: मुझे भूख लगी है",
]
enc = tok(tests, return_tensors="pt", padding=True).to("cuda")
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=60, num_beams=4, no_repeat_ngram_size=3)
for s, o in zip(tests, tok.batch_decode(out, skip_special_tokens=True)):
    print(f"{s}\n  → {o}\n")

trainer_state.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

Saved checkpoint step: 13000
Best validation loss: 2.7157


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

translate English to Tamil: How are you?
  → எப்படி நீங்கள்?

translate English to Spanish: Where is the train station?
  → ¿Dónde está la estación de train?

translate English to Russian: I am hungry.
  → Я hungry.

translate English to Arabic: Thank you very much.
  → شكراً لك كثيراً

translate English to Telugu: Good morning.
  → బాగా రాత్రి.

translate Spanish to English: ¿Dónde está la estación de tren?
  → Where's the train station?

translate Russian to English: Я люблю программирование.
  → I love the program.

translate Hindi to English: मुझे भूख लगी है
  → I'm sick.



## Publish the final model

In [2]:
model.push_to_hub(repo_id, commit_message="Final model: checkpoint 13000 (95% of 1 epoch), val loss 2.716")
tok.push_to_hub(repo_id)
print(f"Official model: https://huggingface.co/{repo_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1dpwrqd/model.safetensors:   0%|          | 11.6kB / 1.20GB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp82d1cq7p/tokenizer.json:  98%|#########8| 16.1MB / 16.3MB            

Validating                    :           |   0%            

Official model: https://huggingface.co/Gunal13/mt5-small-11lang


## Evaluate all 11 languages

In [3]:
!pip install -q datasets sacrebleu

import sacrebleu
import pandas as pd
from datasets import load_dataset

LANGS = {  # code: (OPUS-100 name, language name)
    "hi": ("en-hi", "Hindi"),     "mr": ("en-mr", "Marathi"),
    "ta": ("en-ta", "Tamil"),     "kn": ("en-kn", "Kannada"),
    "ml": ("en-ml", "Malayalam"), "te": ("en-te", "Telugu"),
    "zh": ("en-zh", "Chinese"),   "ar": ("ar-en", "Arabic"),
    "ru": ("en-ru", "Russian"),   "es": ("en-es", "Spanish"),
    "af": ("af-en", "Afrikaans"),
}
N_TEST = 500

def translate_all(sources, batch=32):
    preds = []
    for i in range(0, len(sources), batch):
        enc = tok(sources[i:i + batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=128).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=128, num_beams=4,
                                 no_repeat_ngram_size=3)
        preds += tok.batch_decode(out, skip_special_tokens=True)
    return preds

rows = []
for code, (config, name) in LANGS.items():
    test = load_dataset("Helsinki-NLP/opus-100", config, split="test").select(range(N_TEST))
    en = [t["en"] for t in test["translation"]]
    xx = [t[code] for t in test["translation"]]

    to_xx = translate_all([f"translate English to {name}: " + s for s in en])
    to_en = translate_all([f"translate {name} to English: " + s for s in xx])

    rows.append({
        "Language": name,
        "EN→X chrF": round(sacrebleu.corpus_chrf(to_xx, [xx]).score, 1),
        "X→EN chrF": round(sacrebleu.corpus_chrf(to_en, [en]).score, 1),
        "EN→X BLEU": round(sacrebleu.corpus_bleu(to_xx, [xx],
                           tokenize="zh" if code == "zh" else "13a").score, 1),
        "X→EN BLEU": round(sacrebleu.corpus_bleu(to_en, [en]).score, 1),
    })
    print(f"Done: {name}")

results = pd.DataFrame(rows).sort_values("X→EN chrF", ascending=False)
print("\n=== 11-language results (500 test sentences per direction) ===")
print(results.to_string(index=False))
print("\nAverage chrF:  EN→X", round(results["EN→X chrF"].mean(), 1),
      "  X→EN", round(results["X→EN chrF"].mean(), 1))

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-hi/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  259kB            

en-hi/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-hi/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 65.2MB            

en-hi/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-hi/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  247kB            

... (101 outputs left out)

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Done: Afrikaans

=== 11-language results (500 test sentences per direction) ===
 Language  EN→X chrF  X→EN chrF  EN→X BLEU  X→EN BLEU
  Spanish       39.9       44.5       15.7       20.5
Afrikaans       34.8       43.6       18.3       26.4
  Marathi       30.1       41.8       11.1       23.0
  Kannada       28.2       38.5       12.9       18.0
  Russian       29.3       38.4       10.4       16.4
  Chinese       16.8       37.5       15.9       13.1
   Arabic       24.3       33.5        3.8       12.0
    Hindi       27.0       32.4       10.0       12.5
   Telugu       27.8       32.2       12.4       16.6
    Tamil       29.5       28.6        6.1       10.9
Malayalam       23.1       25.2        1.4        8.3

Average chrF:  EN→X 28.3   X→EN 36.0
